# NLP Basics: Tokenization, POS, NER, WSD

Demonstrating four core NLP tasks using **NLTK** and **spaCy**.

Example sentence: *"John gave Mary two apples at school on Monday."*

## 0. Setup

Install packages (run once) and download required models/corpora.

In [1]:
# Run once — install if not already installed
# !pip install nltk spacy
# !python -m spacy download en_core_web_sm

import nltk
for pkg in [
    'punkt', 'punkt_tab', 'averaged_perceptron_tagger',
    'averaged_perceptron_tagger_eng', 'maxent_ne_chunker',
    'maxent_ne_chunker_tab', 'words', 'wordnet', 'omw-1.4'
]:
    nltk.download(pkg, quiet=True)

import spacy
nlp = spacy.load('en_core_web_sm')

# SENT = "John gave Mary two apples at school on Monday."
SENT = "She is better today, but felt worse yesterday after running."
print(SENT)

She is better today, but felt worse yesterday after running.


### What the setup cell does

It bulk-downloads the NLTK data files (corpora + trained models) the notebook needs. NLTK ships as a small library; the actual data lives in separate packages fetched on demand via `nltk.download(...)`. `quiet=True` suppresses the progress chatter. Files are saved to `~/nltk_data/` (or another path in `nltk.data.path`).

#### What each NLTK package is for

| Package | Purpose |
|---|---|
| `punkt` | Sentence + word tokenizer models (`sent_tokenize`, `word_tokenize`). |
| `punkt_tab` | Newer table-format Punkt data required by NLTK ≥ 3.8.2. Without it `word_tokenize` raises `LookupError` on recent versions. |
| `averaged_perceptron_tagger` | The trained POS tagger model used by `pos_tag`. |
| `averaged_perceptron_tagger_eng` | English-specific tagger resource required by newer NLTK versions (split out from the generic one). |
| `maxent_ne_chunker` | Maximum-entropy chunker model used by `ne_chunk` for NER. |
| `maxent_ne_chunker_tab` | Table-format companion needed by newer NLTK versions of `ne_chunk`. |
| `words` | English word list — used by the NER chunker as a vocabulary lookup. |
| `wordnet` | The WordNet lexical database — needed for `lesk` and `wn.synsets(...)`. |
| `omw-1.4` | Open Multilingual WordNet 1.4 — required since NLTK 3.6 to load WordNet definitions/examples. |

#### Why download them all up front

Each NLP task in this notebook needs different data:

- **Tokenization** → `punkt`, `punkt_tab`
- **POS** → `averaged_perceptron_tagger`, `averaged_perceptron_tagger_eng`
- **NER** → `maxent_ne_chunker`, `maxent_ne_chunker_tab`, `words`
- **WSD** → `wordnet`, `omw-1.4`

Pre-downloading them means later cells run without `LookupError: Resource X not found` surprises.

#### Notes / gotchas

- `quiet=True` only silences output — NLTK still checks each package, but skips re-download if it's current.
- The `_tab` variants are a recent NLTK change; on older NLTK (< 3.8.2) those IDs may print *Package X not found*. Harmless — older NLTK doesn't need them.
- Change install location with `nltk.download(pkg, download_dir='/some/path')`.
- One-shot equivalent: `nltk.download('popular')` grabs a curated bundle, but it's bigger than what's needed here.

## 1. Tokenization

Splitting raw text into tokens (words / punctuation).

In [2]:
# --- NLTK tokenization ---
from nltk.tokenize import word_tokenize, sent_tokenize

nltk_tokens = word_tokenize(SENT)
print('NLTK word tokens :', nltk_tokens)
print('NLTK sentences   :', sent_tokenize("Hello world. NLP is fun! Is it?"))

NLTK word tokens : ['She', 'is', 'better', 'today', ',', 'but', 'felt', 'worse', 'yesterday', 'after', 'running', '.']
NLTK sentences   : ['Hello world.', 'NLP is fun!', 'Is it?']


In [3]:
# --- spaCy tokenization ---
doc = nlp(SENT)
spacy_tokens = [t.text for t in doc]
print('spaCy tokens     :', spacy_tokens)
print('spaCy sentences  :', [s.text for s in nlp("Hello world. NLP is fun! Is it?").sents])

spaCy tokens     : ['She', 'is', 'better', 'today', ',', 'but', 'felt', 'worse', 'yesterday', 'after', 'running', '.']
spaCy sentences  : ['Hello world.', 'NLP is fun!', 'Is it?']


## 2. Part-of-Speech (POS) Tagging

Assigning grammatical category (noun, verb, etc.) to each token.

In [4]:
# --- NLTK POS tagging (Penn Treebank tags) ---
from nltk import pos_tag

nltk_pos = pos_tag(nltk_tokens)
for tok, tag in nltk_pos:
    print(f'  {tok:10s} -> {tag}')

  She        -> PRP
  is         -> VBZ
  better     -> RBR
  today      -> NN
  ,          -> ,
  but        -> CC
  felt       -> VBD
  worse      -> JJR
  yesterday  -> NN
  after      -> IN
  running    -> VBG
  .          -> .


In [5]:
# --- spaCy POS tagging (coarse + fine) + lemma ---
print(f'{"TOKEN":10s} {"LEMMA":10s} {"POS":8s} {"TAG":6s} EXPLANATION')
for t in doc:
    print(f'  {t.text:10s} {t.lemma_:10s} {t.pos_:8s} {t.tag_:6s} {spacy.explain(t.tag_)}')

TOKEN      LEMMA      POS      TAG    EXPLANATION
  She        she        PRON     PRP    pronoun, personal
  is         be         AUX      VBZ    verb, 3rd person singular present
  better     well       ADJ      JJR    adjective, comparative
  today      today      NOUN     NN     noun, singular or mass
  ,          ,          PUNCT    ,      punctuation mark, comma
  but        but        CCONJ    CC     conjunction, coordinating
  felt       feel       VERB     VBD    verb, past tense
  worse      bad        ADJ      JJR    adjective, comparative
  yesterday  yesterday  NOUN     NN     noun, singular or mass
  after      after      ADP      IN     conjunction, subordinating or preposition
  running    run        VERB     VBG    verb, gerund or present participle
  .          .          PUNCT    .      punctuation mark, sentence closer


## 3. Named Entity Recognition (NER)

Identifying real-world entities like persons, places, dates, quantities.

In [6]:
# --- NLTK NER via ne_chunk ---
from nltk import ne_chunk

tree = ne_chunk(nltk_pos)
print(tree)
print('\nExtracted entities:')
for chunk in tree:
    if hasattr(chunk, 'label'):
        print(f'  {chunk.label():12s} {" ".join(c[0] for c in chunk)}')

(S
  She/PRP
  is/VBZ
  better/RBR
  today/NN
  ,/,
  but/CC
  felt/VBD
  worse/JJR
  yesterday/NN
  after/IN
  running/VBG
  ./.)

Extracted entities:


In [7]:
# --- spaCy NER ---
print('Entities found by spaCy:')
for ent in doc.ents:
    print(f'  {ent.text:12s} {ent.label_:10s} ({spacy.explain(ent.label_)})')

Entities found by spaCy:
  today        DATE       (Absolute or relative dates or periods)
  yesterday    DATE       (Absolute or relative dates or periods)


## 4. Word Sense Disambiguation (WSD)

Choosing the correct meaning of a word based on context. We use the **Lesk algorithm** (NLTK) and contrast with spaCy + WordNet lookup.

Example: the word *"bank"* in different contexts.

In [8]:
# --- NLTK WSD with Lesk algorithm ---
from nltk.wsd import lesk
from nltk.corpus import wordnet as wn

sent1 = "I went to the bank to deposit money".split()
sent2 = "The boat reached the bank of the river".split()

for s in (sent1, sent2):
    sense = lesk(s, 'bank', 'n')
    print(f"Sentence : {' '.join(s)}")
    print(f"  sense  : {sense.name() if sense else None}")
    print(f"  defn   : {sense.definition() if sense else None}\n")

Sentence : I went to the bank to deposit money
  sense  : depository_financial_institution.n.01
  defn   : a financial institution that accepts deposits and channels the money into lending activities

Sentence : The boat reached the bank of the river
  sense  : bank.n.01
  defn   : sloping land (especially the slope beside a body of water)



In [ ]:
# All WordNet senses of 'bank'
for i, syn in enumerate(wn.synsets('bank')[:6]): # "Give me all the synonym sets that contain the word bank."
    print(f'{i}. {syn.name():20s} - {syn.definition()}')

0. bank.n.01            - sloping land (especially the slope beside a body of water)
1. depository_financial_institution.n.01 - a financial institution that accepts deposits and channels the money into lending activities
2. bank.n.03            - a long ridge or pile
3. bank.n.04            - an arrangement of similar objects in a row or in tiers
4. bank.n.05            - a supply or stock held in reserve for future use (especially in emergencies)
5. bank.n.06            - the funds held by a gambling house or the dealer in some gambling games


In [ ]:
# --- spaCy doesn't ship WSD out of the box ---
# Common pattern: use spaCy for tokenisation/POS, then run Lesk on each ambiguous token.
from nltk.wsd import lesk

text = "I went to the bank to deposit money"
doc2 = nlp(text)
tokens_for_lesk = [t.text for t in doc2]

for t in doc2:
    if t.pos_ in ('NOUN', 'VERB'):
        sense = lesk(tokens_for_lesk, t.text, t.pos_[0].lower())
        if sense:
            print(f'  {t.text:10s} ({t.pos_}) -> {sense.name()}: {sense.definition()}')

## Summary

| Task          | NLTK                          | spaCy                         |
|---------------|-------------------------------|-------------------------------|
| Tokenization  | `word_tokenize`               | iterate `doc`                 |
| POS tagging   | `pos_tag` (Penn Treebank)     | `token.pos_` / `token.tag_`   |
| NER           | `ne_chunk` (weaker)           | `doc.ents` (stronger)         |
| WSD           | `lesk` + WordNet              | not built-in (use NLTK Lesk)  |